In [ ]:
import json

# Load the JSON file
with open("data/vantage_points/all_anchors_details.json", "r") as file:
    data = json.load(file)

# Extract the list of countries
countries = [details["country"] for details in data.values()]

# Get the unique countries
unique_countries = set(countries)

# Count the unique countries
unique_count = len(unique_countries)

print("Unique Countries:", unique_countries)
print("Number of Unique Countries:", unique_count)


Unique Countries: {'CA', 'MY', 'HU', 'US', 'DE', 'IN', 'BR', 'ZA', 'CL', 'KW', 'AT', 'GB', 'FR', 'MX', 'SA', 'SX', 'VN', 'TW', 'SE', 'NL', 'JP', 'AU', 'IT', 'MZ', 'PL', 'TZ', 'AE', 'HK', 'TR', 'FI', 'KR', 'CO', 'ES', 'SG'}
Number of Unique Countries: 34


In [ ]:
import json
from collections import Counter

# Load the JSON file
with open("data/vantage_points/all_anchors_details.json", "r") as file:
    data = json.load(file)

# Extract ASNs (as_v4 and as_v6) from each anchor
asns = []
for details in data.values():
    if "country" in details and details["country"] is not None:
        asns.append(details["country"])
    # if "as_v6" in details and details["as_v6"] is not None:
    #     asns.append(details["as_v6"])

# Count occurrences of each ASN
asn_counts = Counter(asns)

# Find the ASNs with the most anchors
most_common_asns = asn_counts.most_common(10)  # Top 10 ASNs

# Print the results
print("Top 10 ASNs with the most anchors:")
for asn, count in most_common_asns:
    print(f"ASN {asn}: {count} anchors")


Top 10 ASNs with the most anchors:
ASN US: 10 anchors
ASN AU: 8 anchors
ASN GB: 7 anchors
ASN JP: 6 anchors
ASN BR: 6 anchors
ASN SG: 5 anchors
ASN AE: 2 anchors
ASN DE: 2 anchors
ASN HK: 2 anchors
ASN KR: 2 anchors


In [ ]:
import os
import json
from collections import defaultdict
import csv
import numpy as np
from geoip2.database import Reader
from geopy.distance import geodesic
import requests
from functools import lru_cache


@lru_cache(maxsize=128)
def get_probe_geolocation(probe_id):
    url = f"https://atlas.ripe.net/api/v2/probes/{probe_id}/"
    headers = {"Accept": "application/json"}

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        probe_info = response.json()
        coordinates = probe_info.get("geometry", {}).get("coordinates", None)
        if coordinates:
            return coordinates
        else:
            raise ValueError(f"Coordinates not found for probe ID {probe_id}")
    else:
        response.raise_for_status()


def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """
    # print(lat1, type(lat2), lon1, type(lon2))
    # if np.any([lat1, lon1, lat2, lon2] == None)  :
    #     return np.nan
    if np.any(np.isnan([lat1, lon1, lat2, lon2])):
        return np.nan
    # # Convert latitude and longitude from degrees to radians
    # lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    # # Haversine formula
    # dlat = lat2 - lat1
    # dlon = lon2 - lon1
    # a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    # distance = 6378.1 * 2 * asin(sqrt(a)) # Radius of Earth in kilometers
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers


# Path to your MaxMind GeoIP2 database file
geoip_database_file = "<GEOIP2_CITY_MMDB_PATH>"


def get_ip_info(ip_address):
    if ip_address is None:
        return np.nan, np.nan
    with Reader(geoip_database_file) as reader:
        try:
            response = reader.city(ip_address)
            # country = response.country.name
            # city = response.city.name
            latitude = response.location.latitude
            longitude = response.location.longitude
            if latitude is None or longitude is None:
                return np.nan, np.nan
            return latitude, longitude
        except Exception as e:
            print("Error:", e)
            print(ip_address)
            return np.nan, np.nan


TOTAL_MEASUREMENTS = 120


def compute_completeness_factor(timeline_dict, anchor_id, dst_name):
    # Count only positive values
    positive_values = [
        value for value in timeline_dict[(anchor_id, dst_name)] if value > 0
    ]

    # Calculate the completeness factor for a timeline
    total_measurements = TOTAL_MEASUREMENTS
    complete_measurements = len(positive_values)

    completeness_factor = complete_measurements / total_measurements
    timeline_dict[(anchor_id, dst_name)] = positive_values
    return completeness_factor


def compute_statistics(timeline_data):
    # Compute statistics for a single timeline
    if len(timeline_data) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    minimum = np.nanmin(timeline_data)
    q1 = np.nanpercentile(timeline_data, 25)
    median = np.nanmedian(timeline_data)
    average = np.nanmean(timeline_data)
    std_dev = np.nanstd(timeline_data)
    q3 = np.nanpercentile(timeline_data, 75)
    percentile_90 = np.nanpercentile(timeline_data, 90)
    percentile_95 = np.nanpercentile(timeline_data, 95)
    return minimum, q3 - q1, median, average, std_dev, percentile_90, percentile_95


def process_json_file(
    file_path,
    timeline_dict,
    theoretical_rtt_dict,
    distance_dict,
    anchor_probe_info,
    theoretical_distance_dict_v2,
    theoretical_rtt_dict_v2,
    same_country_dict,
):
    with open(file_path, "r") as file:
        data = json.load(file)

    for measurement in data:
        prb_id = measurement.get("prb_id")
        min_value = measurement.get("min")
        dst_name = measurement.get("dst_name")
        src_addr = measurement.get("src_addr")
        dst_addr = measurement.get("dst_addr")

        # Check if prb_id is present and corresponds to anchor_id
        for anchor_id, stored_prb_id in anchor_probe_info.items():
            if stored_prb_id == prb_id:
                timeline_dict[(anchor_id, dst_name)].append(min_value)
                if theoretical_rtt_dict[(anchor_id, dst_name)] == 0:
                    # Get country for src and dst
                    try:
                        src_lat, src_lon = get_ip_info(src_addr)
                        dst_lat, dst_lon = get_ip_info(dst_addr)

                        # Get countries for source and destination
                        with Reader(geoip_database_file) as reader:
                            src_country = (
                                reader.city(src_addr).country.name if src_addr else None
                            )
                            dst_country = (
                                reader.city(dst_addr).country.name if dst_addr else None
                            )

                        # Check if source and destination are in the same country
                        same_country_dict[(anchor_id, dst_name)] = (
                            src_country == dst_country
                        )

                        # Distance and theoretical RTT calculations
                        distance = haversine(src_lat, src_lon, dst_lat, dst_lon)
                        distancev2 = haversine(
                            *get_probe_geolocation(prb_id), dst_lat, dst_lon
                        )

                        theoretical_rtt_dict[(anchor_id, dst_name)] = (
                            3 * distance * (1 / 299.792458)
                        )
                        distance_dict[(anchor_id, dst_name)] = distance
                        theoretical_distance_dict_v2[(anchor_id, dst_name)] = distancev2
                        theoretical_rtt_dict_v2[(anchor_id, dst_name)] = (
                            3 * distancev2 * (1 / 299.792458)
                        )

                    except Exception as e:
                        print(f"Error processing {src_addr} or {dst_addr}: {e}")


In [ ]:
from collections import Counter

continent_counts = Counter()


In [ ]:
# Get continents for source and destination
with Reader(geoip_database_file) as reader:
    src_continent = reader.city(src_addr).continent.code if src_addr else None
    dst_continent = reader.city(dst_addr).continent.code if dst_addr else None

# Increment timeline count for each continent
if src_continent:
    continent_counts[src_continent] += 1
if dst_continent and dst_continent != src_continent:
    continent_counts[dst_continent] += 1


In [ ]:
# Write results to a CSV for same-country information
with open("data/csv/same_country_timelines.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Source (Anchor ID)", "Destination (Domain Name)", "Same Country"])
    for (anchor_id, dst_name), same_country in same_country_dict.items():
        writer.writerow([anchor_id, dst_name, same_country])

# Print timeline counts per continent
print("Timelines per continent:")
for continent, count in continent_counts.items():
    print(f"{continent}: {count}")


In [ ]:
if __name__ == "__main__":
    input_folder = "data/raw"
    output_file = "data/text/compute_statistics.json"
    anchor_probe_info_file = "data/anchor_probe_info/anchor_probe_info.json"

    timeline_dict = defaultdict(list)
    theoretical_rtt_dict = defaultdict(float)
    distance_dict = defaultdict(float)
    theoretical_rtt_dict_v2 = defaultdict(float)
    theoretical_distance_dict_v2 = defaultdict(float)
    same_country_dict = {}
    continent_counts = Counter()

    # Counters for requested statistics
    same_country_count = 0
    different_country_count = 0
    na_eu_count = 0

    # Load probe to anchor mappings
    with open(anchor_probe_info_file, "r") as anchor_probe_info_file:
        anchor_probe_info = json.load(anchor_probe_info_file)

    # Process files
    for filename in os.listdir(input_folder):
        if filename.endswith(".json"):
            file_path = os.path.join(input_folder, filename)
            process_json_file(
                file_path,
                timeline_dict,
                theoretical_rtt_dict,
                distance_dict,
                anchor_probe_info,
                theoretical_distance_dict_v2,
                theoretical_rtt_dict_v2,
                same_country_dict,
            )

    # Write same-country timelines to CSV
    with open("data/csv/same_country_timelines.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            ["Source (Anchor ID)", "Destination (Domain Name)", "Same Country"]
        )
        for (anchor_id, dst_name), same_country in same_country_dict.items():
            writer.writerow([anchor_id, dst_name, same_country])

    # Print timeline counts per continent
    print("Timelines per continent:")
    for continent, count in continent_counts.items():
        print(f"{continent}: {count}")

    # Print the requested statistics
    print(f"(b) Number of pairs in the same country: {same_country_count}")
    print(f"(c) Number of pairs in different countries: {different_country_count}")
    print(f"(d) Number of pairs in North America - Europe: {na_eu_count}")
